# Notebook 08: Detección en Tiempo Real (Webcam)

Este notebook integra tu cámara web y el detector YOLO para realizar inferencias en vivo en una ventana de OpenCV.

In [ ]:
import sys
import os
from pathlib import Path
import cv2

sys.path.append(os.path.abspath('..'))
from configs import settings
from services.detector import TrafficSignDetector
from utils.camera import CameraManager
from utils.drawing import draw_detections, draw_fps
from utils.fps import FPSCounter

In [ ]:
model_path = Path(settings.MODELS_DIR) / "best.pt"
detector = TrafficSignDetector(model_path if model_path.exists() else settings.MODEL_NAME)
camera = CameraManager(settings.CAMERA_INDEX, settings.FRAME_WIDTH, settings.FRAME_HEIGHT)
fps_counter = FPSCounter()

print("Presiona 'q' en la ventana emergente de OpenCV para salir del bucle en tiempo real.")

while camera.is_opened():
    ret, frame = camera.read_frame()
    if not ret:
        print("⚠️ Falló la lectura del cuadro de la cámara.")
        break
        
    # Detección
    detections = detector.detect(frame)
    fps_counter.tick()
    
    # Dibujo
    annotated_frame = draw_detections(frame, detections)
    annotated_frame = draw_fps(annotated_frame, fps_counter.get_fps())
    
    cv2.imshow("YOLOv8s Detector de Senales de Transito", annotated_frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
        
camera.release()
cv2.destroyAllWindows()
print("Cámara liberada y ventanas destruidas.")